<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Data Interoperability with Unity Catalog</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">4. Centralized Governance with UC with External Compute Engines</span>
    </div>
</div>

<p style="font-size: 1em; text-align: center; line-height: 0; padding-top: 9px; margin: 4px 0">
<img
src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
alt="Databricks Learning"
>
</div>

# 4.2 Demo Processing UC Managed Iceberg Tables with AWS EMR

This demo walks through configuring an AWS EMR cluster running Spark to <b>write</b> to a Unity Catalog managed Iceberg table - the same <code>store_sales_iceberg</code> table built in [3.2 Demo](../3.%20Centralized%20Data%20Processing%20with%20External%20Analytics/3.2%20Demo%20-%20Powering%20Downstream%20Analytics%20in%20Snowflake%20using%20UC%20Managed%20Tables.ipynb) - via the UC Iceberg REST endpoint. Where 3.2 showed an external engine <i>reading</i> UC managed tables, this demo shows external <i>write</i>: same governance, different compute, all credentials vended at job run time.

## Learning Objectives

By the end of this demonstration, you will be able to:
- Identify the UC privileges an external compute engine needs to write to managed Iceberg tables (`MODIFY` + `EXTERNAL USE SCHEMA`)
- Provision a serverless Spark runtime on AWS using an EMR Serverless application
- Configure a Spark job to authenticate to UC via OAuth (service principal client credentials) and use UC as an Iceberg REST catalog
- Verify the EMR-side mutations from Databricks - same source of truth across compute engines

<div style="font-size: 1em; border-left: 4px solid #7b1fa2; background: #f3e5f5; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #4a148c; font-size: 1.1em;">Video Demonstration</strong>
            <p style="margin: 8px 0 0 0; color: #333;">In the standard classroom environment, this demo is delivered as a <strong>video walkthrough</strong> because it requires an AWS account with permissions to provision EMR clusters, a Databricks service principal with OAuth credentials, and customer-managed S3 storage, none of which are available in the lab environment. The notebook below contains the complete working demo. Databricks-side SQL runs in this notebook; AWS-side steps (EMR provisioning, Spark job submission) are shown as reference code blocks. If you have the required infrastructure, you can run it end-to-end.</p>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Required Permissions</strong>
            <p style="margin: 8px 0 0 0; color: #333;">This demo provisions account-level resources on both platforms. The person running it needs:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><b>Databricks:</b> <b>Metastore Admin</b> (External Data Access already enabled from <a href="../3.%20Centralized%20Data%20Processing%20with%20External%20Analytics/3.2%20Demo%20-%20Powering%20Downstream%20Analytics%20in%20Snowflake%20using%20UC%20Managed%20Tables.ipynb">3.2 Demo</a> step A2) and <b>Account Admin</b> (to create a dedicated <code>emr-integration</code> service principal and generate its OAuth secret).</li>
                <li><b>AWS:</b> permissions to create an EMR Serverless application and a job-execution IAM role (typically <code>AdministratorAccess</code>, or scoped equivalents for <code>emr-serverless:*</code>, <code>iam:CreateRole</code>, and <code>s3:*</code> on the script-staging bucket).</li>
            </ul>
        </div>
    </div>
</div>

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Requires Customer-Managed Storage</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The target UC table must live on a <b>customer-managed external location</b> (an S3 path under <i>your own</i> AWS account, registered to UC via a storage credential). EMR Spark needs to read and write the underlying parquet files using credentials vended by UC at job-run time - and those credentials only resolve against buckets you control. Databricks-managed serverless storage (e.g. <code>s3://dbstorage-prod-*</code>) cannot be vended to external IAM principals.</p>
            <p style="margin: 8px 0 0 0; color: #333;">This demo writes to the <code>instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg</code> table. The catalog itself (<code>instructor_interop_demo</code>) must already exist with a customer-managed <code>MANAGED LOCATION</code>. The schema and target table are dropped and recreated by <code>Classroom-Setup-4</code> on every run, so each iteration of the demo starts from a known, deterministic baseline.</p>
        </div>
    </div>
</div>

## A. What We're Building

A managed UC Iceberg table on the Databricks side is exposed through the Unity Catalog Iceberg REST endpoint. An EMR cluster running Spark authenticates to UC via OAuth client credentials (a dedicated `emr-integration` service principal), receives vended S3 credentials, and runs `INSERT` and `UPDATE` mutations against the managed Iceberg table. UC governance, lineage, and audit capture the EMR-side activity exactly as if the writes had come from Databricks.

<details id="what-were-building-details">
  <summary style="cursor: pointer; list-style: none; user-select: none">
    <div style="border-left: 4px solid #1B5162; background: transparent; padding: 16px 20px; border-radius: 4px; margin: 16px 0">
      <div style="display: flex; align-items: center; gap: 12px">
        <span>&#x25B6;</span>
        <strong style="font-size: 1.1em; color: #1B5162">Expand to see diagram</strong>
      </div>
    </div>
  </summary>
  <div style="border-left: 4px solid #1B5162; background: transparent; padding: 0 20px 16px 20px; border-radius: 0 0 4px 4px; margin: -16px 0 16px 0">
    <div style="display: flex; align-items: flex-start; gap: 12px">
      <span style="visibility: hidden">&#x25B6;</span>
      <div style="width: 100%">
<div class="mermaid" id="diagram-4-2-emr-architecture" style="font-size: 1em;">
flowchart TB
    subgraph DBX["Databricks (Unity Catalog)"]
        direction TB
        ICE["<b>store_sales_iceberg</b><br/><i>Iceberg Managed Table<br/>(customer-managed S3)</i>"]
    end
    REST["<b>Iceberg REST Catalog</b><br/><i>External Data Access enabled</i>"]
    subgraph AWS["AWS"]
        direction TB
        CLUSTER["<b>EMR Cluster (single-node)</b><br/><i>Spark 3.5 + Iceberg runtime</i>"]
        SPARK["<b>spark-shell / spark-submit</b><br/><i>INSERT / UPDATE<br/>against UC catalog</i>"]
        CLUSTER --> SPARK
    end
    ENGINEER["<b>Data Engineer</b><br/><i>SSH to master, run script</i>"]
    ICE  --> REST
    REST -- "SP / OAuth<br/>vended creds" --> AWS
    ENGINEER  --> AWS
    SPARK -- "writes back" --> ICE
    style DBX fill:#fff5f3,stroke:#FF3621,stroke-width:2px
    style ICE fill:#ffffff,stroke:#FF3621
    style REST fill:#FF3621,stroke:#CC2B1A,stroke-width:2px,color:#fff
    style AWS fill:#fff8f0,stroke:#FF9900,stroke-width:2px
    style CLUSTER fill:#ffffff,stroke:#FF9900
    style SPARK fill:#ffffff,stroke:#FF9900
    style ENGINEER fill:#eceff1,stroke:#37474f,stroke-width:2px
</div>
      </div>
    </div>
  </div>
</details>
<script type="module">
import mermaid from "https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.esm.min.mjs";
mermaid.initialize({ startOnLoad: false });
const id = "#diagram-4-2-emr-architecture";
async function renderDiagram() {
  await mermaid.run({ querySelector: id });
  document.querySelectorAll(id + ' svg text, ' + id + ' svg .nodeLabel, ' + id + ' svg foreignObject div, ' + id + ' svg span').forEach(el => { el.style.fontSize = '1em'; });
}
await new Promise(r => requestAnimationFrame(r));
try { await renderDiagram(); } catch(e) { await new Promise(r => setTimeout(r, 1000)); await renderDiagram(); }
const det = document.getElementById('what-were-building-details');
if (det) {
  det.addEventListener('toggle', async () => {
    if (!det.open) return;
    const node = document.querySelector(id);
    if (node && !node.querySelector('svg')) {
      node.removeAttribute('data-processed');
      await renderDiagram();
    }
  });
}
</script>

In [0]:
%run ../Includes/Classroom-Setup-4

## B. Databricks Preparation

The Databricks-side setup happens in this notebook: confirm the target table is in place, enable external data access on the metastore (one-time, may be already done), create and configure the `emr-integration` service principal that EMR will authenticate as, and grant it the UC privileges to read and write the target table.

### B1. (Databricks) Verify the UC Iceberg Target Exists

`Classroom-Setup-4` (above) has just dropped and recreated `store_sales_iceberg` so the demo starts from a known baseline. Confirm it landed on customer-managed storage and check the row count for the slice the EMR mutations will touch.

In [0]:
SELECT
  table_name,
  data_source_format,
  table_type,
  storage_path,
  (SELECT COUNT(*) FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg)
    AS total_rows,
  (SELECT COUNT(*) FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg
    WHERE ss_sold_date_sk = 2450816
      AND ss_item_sk BETWEEN 1000 AND 1010)
    AS mutation_slice_rows
FROM system.information_schema.tables
WHERE table_catalog = 'instructor_interop_demo'
  AND table_schema  = 'data_interoperability_tpcds'
  AND table_name    = 'store_sales_iceberg';

### B2. (Databricks) Enable External Data Access on the Metastore

This is a one-time metastore-level toggle that exposes the Unity Catalog Iceberg REST endpoint to external clients. If you completed [3.2 Demo step A2](../3.%20Centralized%20Data%20Processing%20with%20External%20Analytics/3.2%20Demo%20-%20Powering%20Downstream%20Analytics%20in%20Snowflake%20using%20UC%20Managed%20Tables.ipynb), this is already done and you can skip to B3. Otherwise, follow the steps in the pulldown below.

<div style="font-size: 1em; border-left: 4px solid #ff9800; background: #fff3e0; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #e65100; font-size: 1.1em;">Metastore Admin Required</strong>
            <p style="margin: 8px 0 0 0; color: #333;">The External Data Access toggle lives in <strong>Catalog Explorer -&gt; gear icon -&gt; Metastore</strong> and is only visible to Metastore Admins. If you do not see it, your account does not have the role.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Enable External Data Access (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Open <strong>Catalog Explorer</strong></li>
      <li>Click the &#9881; icon (top right of the catalog tree) and choose <strong>Metastore</strong></li>
      <li>In the metastore details page, toggle <strong>External data access</strong> to <strong>enabled</strong></li>
    </ol>
  </div>
</details>

### B3. (Databricks) Create the `emr-integration` Service Principal

Create a new account-level service principal that EMR will authenticate as. Keeping this SP separate from the Snowflake SP from 3.2 means we can revoke EMR's access independently and the audit log clearly attributes EMR-side activity to a distinct identity.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Create emr-integration SP (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Click the user avatar (top right of the workspace) -&gt; <strong>Settings</strong></li>
      <li>Open <strong>Identity and Access</strong> -&gt; <strong>Service principals</strong> -&gt; <strong>Manage</strong></li>
      <li>Click <strong>Add service principal</strong> -&gt; <strong>Add new</strong></li>
      <li>Enter the display name <code>emr-integration</code> and create</li>
      <li>From the service principal detail page, copy the <strong>Application ID</strong> - you will need it in B6 (the GRANT statements) and in C4 (the PySpark driver script)</li>
    </ol>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Show Screenshot (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <img src="https://files.training.databricks.com/binder/prod_main/data-interoperability-with-unity-catalog-en_us-2.1.0/images/20260915T185754Z/Data Interoperability with Unity Catalog/Includes/images/emr-create-service-principal.png" alt="Workspace Settings - Identity and Access - Service principals page showing the emr-integration SP created" style="max-width: 100%; height: auto; border: 1px solid #d0d0d0; border-radius: 4px;"/>
  </div>
</details>

### B4. (Databricks) Add the Service Principal to the Workspace

An account-level SP is not automatically a member of any workspace - and the workspace's OIDC token endpoint will reject its credentials with `invalid_client` until it is federated. This step enrols the existing account SP as a workspace user. After this, the OAuth client credentials flow against this workspace will start succeeding immediately.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Add SP to Workspace (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>Click the user avatar -&gt; <strong>Settings</strong></li>
      <li>Open <strong>Identity and Access</strong> -&gt; <strong>Users</strong> -&gt; <strong>Manage</strong></li>
      <li>Click <strong>Add user</strong></li>
      <li>Choose <strong>Add existing</strong> and search for <code>emr-integration</code> by name or App ID</li>
      <li>Select it and click <strong>Add</strong></li>
    </ol>
    <p style="margin: 12px 0 0 0; color: #333;">The SP should now appear in the workspace's Users list.</p>
  </div>
</details>

### B5. (Databricks) Generate an OAuth Secret for the Service Principal

The catalog-credentials flow that EMR uses needs the SP's Application ID (from B3) plus a client secret generated here.

<div style="font-size: 1em; border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #c62828; font-size: 1.1em;">Treat This Secret Like a Password</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Never paste it into a notebook cell, a chat, or a ticket. Store it in a password manager or secrets vault, and rotate it on a schedule. The PySpark driver in C4 reads it from a runtime variable - in production it should come from AWS Secrets Manager or a similar vault, not the command line.</p>
        </div>
    </div>
</div>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.simpleicons.org/databricks/FF3621" width="20" height="20" style="vertical-align: middle;"> Databricks:</span> Generate OAuth Secret (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <p style="margin: 0 0 12px 0; color: #333;">In the Databricks workspace UI:</p>
    <ol style="margin: 0 0 0 20px; color: #333;">
      <li>On the <code>emr-integration</code> service principal detail page, open the <strong>Secrets</strong> tab</li>
      <li>Click <strong>Generate secret</strong>, choose a <strong>lifetime</strong> (rotate before expiry), then <strong>Generate</strong></li>
      <li><strong>Copy the secret immediately</strong> - it is shown exactly once</li>
    </ol>
  </div>
</details>

### B6. (Databricks) Grant UC Privileges to the Service Principal

The SP now exists and can authenticate, but holds no UC privileges yet. The cell below grants the privileges EMR needs to read and write the target table:

- `EXTERNAL USE SCHEMA` on the catalog (lets the SP use the Iceberg REST endpoint and receive vended storage credentials)
- `USE CATALOG` and `USE SCHEMA` (standard traversal)
- `SELECT` on the target table (read)
- `MODIFY` on the target table (write - the new privilege EMR specifically needs)

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Before Running</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Replace the <code>emr_sp_app_id</code> in the cell below with the Application ID copied from B3.</p>
        </div>
    </div>
</div>

In [0]:
-- Run as Metastore Admin / Account Admin.
-- Replace the Application ID with the id created in step B3.
BEGIN
  DECLARE sp_uuid STRING DEFAULT '7bdd0236-f1a8-46ed-a9d3-398471e83906'; -- replace with your emr_sp_app_id
  DECLARE cat STRING DEFAULT 'instructor_interop_demo';
  DECLARE sch STRING DEFAULT 'data_interoperability_tpcds';

  EXECUTE IMMEDIATE
    'GRANT EXTERNAL USE SCHEMA ON CATALOG `' || cat || '` TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT USE CATALOG ON CATALOG `' || cat || '` TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT USE SCHEMA ON SCHEMA `' || cat || '`.`' || sch || '` TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT SELECT ON TABLE `' || cat || '`.`' || sch || '`.store_sales_iceberg TO `' || sp_uuid || '`';
  EXECUTE IMMEDIATE
    'GRANT MODIFY ON TABLE `' || cat || '`.`' || sch || '`.store_sales_iceberg TO `' || sp_uuid || '`';
END;

In [0]:
-- Confirm the SP now holds SELECT + MODIFY on the target.
-- Replace the Application ID with the id created in step B3.
SHOW GRANTS `7bdd0236-f1a8-46ed-a9d3-398471e83906` -- replace with your emr_sp_app_id
  ON TABLE instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg;

## C. AWS Preparation

A small single-node EMR cluster is enough for this demo. Provision it, wait for `WAITING`, open SSH, then run the PySpark driver. The CLI snippets below are reference - use whatever provisioning method your team already uses.

### C1. (AWS) Create the EMR Cluster

Substitute `{your-keypair}` with the name of an EC2 key pair you can SSH with.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Create EMR Cluster (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
aws emr create-cluster \
  --region ap-southeast-2 \
  --name 'data-interop-demo-cluster' \
  --release-label emr-7.5.0 \
  --applications Name=Spark Name=Hadoop \
  --instance-type m5.xlarge \
  --instance-count 1 \
  --use-default-roles \
  --ec2-attributes KeyName={your-keypair} \
  --auto-termination-policy IdleTimeout=10800
<br/>
# Capture the returned ClusterId (j-XXXXXXXX) - C2 and E1 need it.
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'bash';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'python' ? 'PySpark' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### C2. (AWS) Wait for the Cluster, Then Open SSH

Wait for `WAITING`, then add an inbound TCP/22 rule on the master security group - EMR does not open port 22 by default.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Wait for Cluster Ready (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
# Re-run until State == WAITING (about 5-8 minutes), or use the wait command.
aws emr describe-cluster \
  --region ap-southeast-2 \
  --cluster-id {cluster-id} \
  --query 'Cluster.{state:Status.State,master_dns:MasterPublicDnsName}' \
  --output table
<br/>
aws emr wait cluster-running \
  --region ap-southeast-2 \
  --cluster-id {cluster-id}
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Authorize SSH Ingress (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
MASTER_SG=$(aws emr describe-cluster \
  --region ap-southeast-2 \
  --cluster-id {cluster-id} \
  --query 'Cluster.Ec2InstanceAttributes.EmrManagedMasterSecurityGroup' \
  --output text)
<br/>
aws ec2 authorize-security-group-ingress \
  --region ap-southeast-2 \
  --group-id "$MASTER_SG" \
  --protocol tcp \
  --port 22 \
  --cidr 0.0.0.0/0
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'bash';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'python' ? 'PySpark' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### C3. (AWS) SSH and Launch PySpark

SSH in, then start a `pyspark` REPL with the Iceberg + AWS SDK packages on the classpath. The `master-public-dns` came from C2's `describe-cluster` output.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> SSH and Launch PySpark (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
ssh -i ~/.ssh/{your-keypair}.pem hadoop@{master-public-dns}
<br/>
# First launch resolves Maven artifacts - expect ~60-90s before the prompt appears.
pyspark \
  --packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,software.amazon.awssdk:bundle:2.25.43,software.amazon.awssdk:url-connection-client:2.25.43
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'bash';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'python' ? 'PySpark' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

### C4. (AWS) PySpark Driver Script

Set environment variables for the four config values, then run the PySpark script. The script lets the Iceberg REST client manage tokens itself - it runs an OAuth2 client-credentials flow against the workspace OIDC endpoint and refreshes on expiry.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Set Environment Variables (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
# Set the variables the PySpark script reads. Keep them in your shell
# session only - do NOT commit secrets to source.
# Replace the values below with your values.
<br/>
export DBX_WORKSPACE_HOST="dbc-82c32078-51f7.cloud.databricks.com"
export DBX_CATALOG="instructor_interop_demo"
export DBX_SP_CLIENT_ID="7bdd0236-f1a8-46ed-a9d3-398471e83906"
export DBX_SP_CLIENT_SECRET="****"
    </div>
  </div>
</details>

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> PySpark Driver Script (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="python">
# === EMR PySpark driver script ===
# Registers UC as an Iceberg REST catalog and runs INSERT and UPDATE
# against a UC managed Iceberg table. Run from a pyspark / spark-shell on the
# EMR master, or save as mutate_uc_table.py and spark-submit it.
#
# Reads its configuration from environment variables - export them first
# (see the previous pulldown) so no credentials are written to disk.
<br/>
import os
from pyspark.sql import SparkSession
<br/>
# 1. Load configuration from env vars and fail fast if anything is missing.
REQUIRED = [
    "DBX_WORKSPACE_HOST",
    "DBX_CATALOG",
    "DBX_SP_CLIENT_ID",
    "DBX_SP_CLIENT_SECRET",
]
missing = [name for name in REQUIRED if not os.environ.get(name)]
if missing:
    raise SystemExit(f"Missing required environment variables: {', '.join(missing)}")
<br/>
WORKSPACE_HOST   = os.environ["DBX_WORKSPACE_HOST"]
CATALOG          = os.environ["DBX_CATALOG"]
SP_CLIENT_ID     = os.environ["DBX_SP_CLIENT_ID"]
SP_CLIENT_SECRET = os.environ["DBX_SP_CLIENT_SECRET"]
<br/>
# 2. Configure UC as an Iceberg REST catalog. Iceberg runs the OAuth2
#    client-credentials flow itself against the workspace OIDC endpoint,
#    refreshes the token on expiry. The SP must be federated to the workspace
#    (B4) for the workspace endpoint to accept its credentials.
spark.conf.set("spark.sql.catalog.uc", "org.apache.iceberg.spark.SparkCatalog")
spark.conf.set("spark.sql.catalog.uc.type", "rest")
spark.conf.set("spark.sql.catalog.uc.uri", f"https://{WORKSPACE_HOST}/api/2.1/unity-catalog/iceberg-rest")
spark.conf.set("spark.sql.catalog.uc.warehouse", CATALOG)
spark.conf.set("spark.sql.catalog.uc.oauth2-server-uri", f"https://{WORKSPACE_HOST}/oidc/v1/token")
spark.conf.set("spark.sql.catalog.uc.credential", f"{SP_CLIENT_ID}:{SP_CLIENT_SECRET}")
spark.conf.set("spark.sql.catalog.uc.scope", "all-apis sql")
spark.conf.set("spark.sql.catalog.uc.header.X-Iceberg-Access-Delegation", "vended-credentials")
spark.conf.set("spark.sql.catalog.uc.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
spark.conf.set("spark.sql.defaultCatalog", "uc")
<br/>
# The catalog is already pinned via spark.sql.catalog.uc.warehouse above,
# so the namespace path here is schema-only - sending {catalog}.{schema}
# would be interpreted as a nested namespace and rejected by the REST endpoint.
spark.sql("USE uc.data_interoperability_tpcds")
<br/>
# 3a. INSERT - duplicate a small slice of the table back onto itself so the
#     row count visibly grows when verified from Databricks.
spark.sql("""
    INSERT INTO store_sales_iceberg
    SELECT * FROM store_sales_iceberg
    WHERE ss_sold_date_sk = 2450816 AND ss_item_sk BETWEEN 1000 AND 1010
""")
<br/>
# 3b. UPDATE - bump prices on a single date+item slice by 5%.
spark.sql("""
    UPDATE store_sales_iceberg
       SET ss_sales_price = CAST(ss_sales_price * 1.05 AS DECIMAL(7,2))
     WHERE ss_sold_date_sk = 2450816
       AND ss_item_sk      = 1000
""")
<br/>
print("Mutations complete.")
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'bash';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'python' ? 'PySpark' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## D. Databricks Verification

Once the EMR Serverless job reports `SUCCESS`, the same `store_sales_iceberg` table queried from Databricks reflects the EMR-side changes. UC governance, lineage, and audit captured the operations regardless of which engine ran them.

### D1. (Databricks) Inspect the Commit History

`DESCRIBE HISTORY` reads straight from the table's metadata layer and lists every commit with its operation, parameters, calling principal, and timestamp. Because `Classroom-Setup-4` rebuilt the table at the start of the demo, the versions are deterministic: version 0 is the initial create + populate, and the EMR commits from C4 land at versions 1 (`INSERT`) and 2 (`UPDATE`). Each one is attributed to the `emr-integration` SP - the same governance trail you would see for a Databricks-side write.

In [0]:
SELECT version, operation, engineInfo
FROM (DESCRIBE HISTORY instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg);

<div style="font-size: 1em; border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #0d47a1; font-size: 1.1em;">Why does every commit show <code>operation = WRITE</code>?</strong>
            <p style="margin: 8px 0 0 0; color: #333;">Both EMR commits land with the same operation label, even though C4 ran an <code>INSERT</code> and an <code>UPDATE</code>. This is expected and is a real difference between Iceberg and Delta:</p>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li><strong>Iceberg</strong> defines only four operation values in its <a href="https://iceberg.apache.org/spec/#snapshots" target="_blank" style="color:#0d47a1; text-decoration: underline">snapshot spec</a>: <code>append</code>, <code>replace</code>, <code>overwrite</code>, <code>delete</code>. The Iceberg Spark runtime maps SQL DML to these primitives - <code>INSERT</code> -&gt; <code>append</code>, <code>UPDATE</code> (file rewrite) -&gt; <code>overwrite</code>, <code>DELETE</code> -&gt; <code>delete</code>. The original SQL verb is not preserved in the snapshot metadata. <code>DESCRIBE HISTORY</code> surfaces these as <code>WRITE</code>.</li>
                <li><strong>Delta</strong> stores the SQL-level operation string directly in its transaction log (<code>WRITE</code>, <code>UPDATE</code>, <code>MERGE</code>, <code>DELETE</code>, <code>OPTIMIZE</code>, ...), so a UC managed table's <code>DESCRIBE HISTORY</code> shows the verb you ran.</li>
            </ul>
            <p style="margin: 8px 0 0 0; color: #333;">There is also no per-commit metric breakdown to fall back on for UC managed Iceberg from Databricks SQL: <code>DESCRIBE HISTORY</code>'s <code>operationMetrics</code> column is populated for Delta but comes back <code>NULL</code> for Iceberg, and Iceberg's native metadata tables (<code>&lt;table&gt;.snapshots</code>, <code>.history</code>, <code>.files</code>) require the Iceberg Spark catalog plugin and aren't reachable through Databricks SQL. To get added/removed file counts and engine info per snapshot, query the table from an external Iceberg client (PyIceberg, or Spark with the Iceberg runtime configured) via the UC Iceberg REST endpoint - the same endpoint EMR used to write in C4.</p>
        </div>
    </div>
</div>

### D2. (Databricks) Walk the Versions with Time Travel

Each EMR commit landed at a known version (1 and 2 from D1). Time-travelling between adjacent versions and joining on the row key gives a quasi-CDF view without needing the table to have CDF enabled. The next two cells walk versions 0&rarr;1 (`INSERT`) and 1&rarr;2 (`UPDATE`). Each query targets the slice EMR touched (`ss_sold_date_sk = 2450816`, `ss_item_sk` between 1000 and 1010).

#### D2a. INSERT (versions 0 → 1)

The PySpark `INSERT` from C4 duplicated the slice back onto itself. Group by `ss_item_sk` at each version and compare row counts - every key in 1000..1010 should show `rows_after = rows_before * 2`.

In [0]:
WITH v0 AS (
  SELECT ss_item_sk, COUNT(*) AS rows_before
  FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg VERSION AS OF 0
  WHERE ss_sold_date_sk = 2450816 AND ss_item_sk BETWEEN 1000 AND 1010
  GROUP BY ss_item_sk
),
v1 AS (
  SELECT ss_item_sk, COUNT(*) AS rows_after
  FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg VERSION AS OF 1
  WHERE ss_sold_date_sk = 2450816 AND ss_item_sk BETWEEN 1000 AND 1010
  GROUP BY ss_item_sk
)
SELECT
  v0.ss_item_sk,
  v0.rows_before,
  v1.rows_after,
  v1.rows_after - v0.rows_before AS rows_inserted
FROM v0 JOIN v1 USING (ss_item_sk)
ORDER BY ss_item_sk;

#### D2b. UPDATE (versions 1 → 2)

The `UPDATE` bumped `ss_sales_price` by 5% for `ss_item_sk = 1000` only. Joining the two versions on `ss_item_sk` and comparing aggregates shows the price change concentrated on that one key; everything else is unchanged.

In [0]:
WITH v1 AS (
  SELECT ss_item_sk,
         ROUND(AVG(ss_sales_price), 4) AS avg_price_before,
         ROUND(MAX(ss_sales_price), 2) AS max_price_before
  FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg VERSION AS OF 1
  WHERE ss_sold_date_sk = 2450816 AND ss_item_sk = 1000
  GROUP BY ss_item_sk
),
v2 AS (
  SELECT ss_item_sk,
         ROUND(AVG(ss_sales_price), 4) AS avg_price_after,
         ROUND(MAX(ss_sales_price), 2) AS max_price_after
  FROM instructor_interop_demo.data_interoperability_tpcds.store_sales_iceberg VERSION AS OF 2
  WHERE ss_sold_date_sk = 2450816 AND ss_item_sk = 1000
  GROUP BY ss_item_sk
)
SELECT
  v1.ss_item_sk,
  v1.avg_price_before,
  v2.avg_price_after,
  ROUND(v2.avg_price_after / v1.avg_price_before, 4) AS price_ratio,
  v1.max_price_before,
  v2.max_price_after
FROM v1 JOIN v2 USING (ss_item_sk)
ORDER BY ss_item_sk;

## E. Teardown

Terminate the EMR cluster when done. EMR-on-EC2 charges per-second for as long as the master node is running, so don't leave it idle.

### E1. (AWS) Terminate the EMR Cluster

Run the AWS CLI command in the pulldown below from your local shell to terminate the cluster created in C1.

<details style="margin: 16px 0; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
  <summary style="padding: 12px 16px; background: #f5f5f5; cursor: pointer; font-weight: 600; font-size: 1em;">
    <span style="white-space: nowrap;"><img src="https://cdn.jsdelivr.net/gh/devicons/devicon@latest/icons/amazonwebservices/amazonwebservices-original-wordmark.svg" width="32" height="20" style="vertical-align: middle;"> AWS:</span> Terminate EMR Cluster (CLI) (click to expand)
  </summary>
  <div style="padding: 16px; background: #fafafa;">
    <div class="code-block" data-language="bash">
# Substitute {cluster-id} with the j-XXXXXXXX from C1.
# This shuts down the master node immediately.
aws emr terminate-clusters \
  --region ap-southeast-2 \
  --cluster-ids {cluster-id}
<br/>
# Verify the state transition (should go RUNNING / WAITING -> TERMINATING -> TERMINATED).
aws emr describe-cluster \
  --region ap-southeast-2 \
  --cluster-id {cluster-id} \
  --query 'Cluster.Status.State'
    </div>
  </div>
</details>

<link href="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/themes/prism.min.css" rel="stylesheet" />
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/prism.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-sql.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-bash.min.js"></script>
<script src="https://cdnjs.cloudflare.com/ajax/libs/prism/1.29.0/components/prism-python.min.js"></script>

<script>
(function() {
    document.querySelectorAll('.code-block').forEach(function(block) {
        if (block.getAttribute('data-processed')) return;
        block.setAttribute('data-processed', 'true');
        var lang = block.getAttribute('data-language') || 'bash';
        var label = lang === 'bash' ? 'Terminal' : (lang === 'python' ? 'PySpark' : 'AWS');
        var code = block.textContent.trim();
        var id = 'code-' + Math.random().toString(36).substr(2, 9);
        block.innerHTML =
            '<div style="position:relative;margin:16px 0;">' +
                '<div style="position:absolute;top:8px;left:12px;font-size:11px;color:#666;font-weight:600;text-transform:uppercase;">' + label + '</div>' +
                '<button class="copy-btn" style="position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#ddd;color:#333;border:1px solid #ccc;border-radius:4px;cursor:pointer;z-index:10;">Copy</button>' +
                '<pre style="background:#f8f8f8;border-radius:8px;padding:16px;padding-top:32px;overflow-x:auto;margin:0;border:1px solid #e0e0e0;"><code id="' + id + '" class="language-' + lang + '" style="font-family:Consolas,Monaco,monospace;font-size:13px;"></code></pre>' +
            '</div>';
        var codeEl = document.getElementById(id);
        codeEl.textContent = code;
        Prism.highlightElement(codeEl);
        block.querySelector('.copy-btn').onclick = function() {
            var t = document.createElement('textarea');
            t.value = code;
            document.body.appendChild(t);
            t.select();
            document.execCommand('copy');
            document.body.removeChild(t);
            this.textContent = '✓ Copied!';
            setTimeout(() => this.textContent = 'Copy', 2000);
        };
    });
})();
</script>

## Key Takeaways

<div style="font-size: 1em; border-left: 4px solid #4caf50; background: #e8f5e9; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
    <div style="display: flex; align-items: flex-start; gap: 12px;">
        <div>
            <strong style="color: #2e7d32; font-size: 1.1em;">What This Demo Shows</strong>
            <ul style="margin: 8px 0 0 16px; color: #333">
                <li>External compute (EMR Serverless Spark) can <strong>read and write</strong> UC managed Iceberg tables through the Iceberg REST endpoint</li>
                <li>Writes require <code>MODIFY</code> on the table on top of the read-side privileges (<code>EXTERNAL USE SCHEMA</code>, <code>USE CATALOG</code>, <code>USE SCHEMA</code>, <code>SELECT</code>)</li>
                <li>EMR Serverless removes the cluster-provisioning ceremony of classic EMR - jobs submit against a long-lived <em>application</em> rather than a running cluster, and AWS scales workers up and down on demand</li>
                <li><strong>The same SP from 3.2</strong> serves both engines - Snowflake reads via OAuth, EMR writes via OAuth, UC governance is the single source of truth</li>
                <li>The <code>store_sales_iceberg</code> table is mutated by EMR but the mutations appear in UC's audit log attributed to the SP - lineage and governance unchanged regardless of which engine ran the workload</li>
            </ul>
        </div>
    </div>
</div>

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>